# Handwriting Verification with Triplet Loss

This notebook demonstrates training a handwriting verification system using triplet loss and comprehensive biometric evaluation.

In [ ]:
import os
import sys
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.append('..')

from src.data import TripletDataset, create_triplet_dataloaders
from src.models import TripletMobileNetV3Small, TripletResNet18
from src.training import TripletTrainer
from src.evaluation import (
    evaluate_comprehensive,
    plot_comprehensive_results,
    plot_training_history_triplet
)
from src.utils import set_seed, get_device

# Set random seed
set_seed(42)

# Get device
device = get_device()

## Configuration

In [ ]:
# Paths
IAM_ROOT = "../datasets/processed-handwritten/iam_processed"
RIMES_ROOT = "../datasets/processed-handwritten/rimes_processed"
RESULTS_DIR = "../results/triplet"

# Hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 4
TARGET_SIZE = 448
EMBEDDING_DIM = 128
MARGIN = 0.5
EPOCHS = 10
PATIENCE = 7
TRIPLETS_PER_WRITER = 100

os.makedirs(RESULTS_DIR, exist_ok=True)

## Load and Split Datasets

In [ ]:
# Load IAM
iam_dirs = [
    os.path.join(IAM_ROOT, d)
    for d in sorted(os.listdir(IAM_ROOT))
    if os.path.isdir(os.path.join(IAM_ROOT, d))
]

# Load RIMES
rimes_dirs = [
    os.path.join(RIMES_ROOT, d)
    for d in sorted(os.listdir(RIMES_ROOT))
    if os.path.isdir(os.path.join(RIMES_ROOT, d))
]

print(f"IAM writers: {len(iam_dirs)}")
print(f"RIMES writers: {len(rimes_dirs)}")

# Split IAM
iam_train, iam_temp = train_test_split(iam_dirs, test_size=0.2, random_state=42)
iam_val, iam_test = train_test_split(iam_temp, test_size=0.5, random_state=42)

# Split RIMES
rimes_train, rimes_temp = train_test_split(rimes_dirs, test_size=0.2, random_state=42)
rimes_val, rimes_test = train_test_split(rimes_temp, test_size=0.5, random_state=42)

print(f"\nIAM: Train={len(iam_train)}, Val={len(iam_val)}, Test={len(iam_test)}")
print(f"RIMES: Train={len(rimes_train)}, Val={len(rimes_val)}, Test={len(rimes_test)}")

## Experiment 1: Train IAM → Test IAM

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 1: Train IAM → Test IAM")
print("="*70)

# Create datasets
train_ds = TripletDataset(iam_train, train=True, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)
val_ds = TripletDataset(iam_val, train=False, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)
test_ds = TripletDataset(iam_test, train=False, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)

# Create dataloader
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

# Create model
model = TripletMobileNetV3Small(embedding_dim=EMBEDDING_DIM)

# Create trainer
trainer = TripletTrainer(
    model=model,
    model_name="iam_to_iam_triplet",
    device=device,
    margin=MARGIN,
    results_dir=RESULTS_DIR
)

# Train
history = trainer.train(
    train_loader=train_loader,
    val_dataset=val_ds,
    epochs=EPOCHS,
    patience=PATIENCE
)

# Plot training history
plot_training_history_triplet(
    history=history,
    model_name="IAM→IAM Triplet",
    save_path=os.path.join(RESULTS_DIR, "iam_to_iam_history.png")
)

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "iam_to_iam_triplet_best.pth")))

metrics = evaluate_comprehensive(
    model=model,
    dataset=test_ds,
    device=device,
    num_pairs=2000,
    dataset_name="IAM→IAM Test"
)

plot_comprehensive_results(
    metrics=metrics,
    save_path=os.path.join(RESULTS_DIR, "iam_to_iam_comprehensive.png")
)

## Experiment 2: Train IAM → Test RIMES (Cross-Dataset)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 2: Train IAM → Test RIMES (Cross-Dataset)")
print("="*70)

# Create RIMES test dataset
test_ds_rimes = TripletDataset(rimes_test, train=False, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)

metrics = evaluate_comprehensive(
    model=model,
    dataset=test_ds_rimes,
    device=device,
    num_pairs=2000,
    dataset_name="IAM→RIMES Test"
)

plot_comprehensive_results(
    metrics=metrics,
    save_path=os.path.join(RESULTS_DIR, "iam_to_rimes_comprehensive.png")
)

## Experiment 3: Train RIMES → Test RIMES

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 3: Train RIMES → Test RIMES")
print("="*70)

# Create datasets
train_ds = TripletDataset(rimes_train, train=True, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)
val_ds = TripletDataset(rimes_val, train=False, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)
test_ds = TripletDataset(rimes_test, train=False, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)

# Create dataloader
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

# Create model
model = TripletMobileNetV3Small(embedding_dim=EMBEDDING_DIM)

# Create trainer
trainer = TripletTrainer(
    model=model,
    model_name="rimes_to_rimes_triplet",
    device=device,
    margin=MARGIN,
    results_dir=RESULTS_DIR
)

# Train
history = trainer.train(
    train_loader=train_loader,
    val_dataset=val_ds,
    epochs=EPOCHS,
    patience=PATIENCE
)

# Plot training history
plot_training_history_triplet(
    history=history,
    model_name="RIMES→RIMES Triplet",
    save_path=os.path.join(RESULTS_DIR, "rimes_to_rimes_history.png")
)

In [ ]:
# Load best model and evaluate
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "rimes_to_rimes_triplet_best.pth")))

metrics = evaluate_comprehensive(
    model=model,
    dataset=test_ds,
    device=device,
    num_pairs=2000,
    dataset_name="RIMES→RIMES Test"
)

plot_comprehensive_results(
    metrics=metrics,
    save_path=os.path.join(RESULTS_DIR, "rimes_to_rimes_comprehensive.png")
)

## Experiment 4: Train RIMES → Test IAM (Cross-Dataset)

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT 4: Train RIMES → Test IAM (Cross-Dataset)")
print("="*70)

# Create IAM test dataset
test_ds_iam = TripletDataset(iam_test, train=False, triplets_per_writer=TRIPLETS_PER_WRITER, target_size=TARGET_SIZE)

metrics = evaluate_comprehensive(
    model=model,
    dataset=test_ds_iam,
    device=device,
    num_pairs=2000,
    dataset_name="RIMES→IAM Test"
)

plot_comprehensive_results(
    metrics=metrics,
    save_path=os.path.join(RESULTS_DIR, "rimes_to_iam_comprehensive.png")
)

## Summary

This notebook demonstrates:
- Training triplet networks for handwriting verification
- Comprehensive biometric evaluation (EER, AUC, d-prime, etc.)
- Cross-dataset generalization testing
- Visualization of distance distributions and ROC curves